# manga-lens: train trên Kaggle
Bật trước: **Internet On**, **GPU T4 x2/P100**, Secret `HF_TOKEN` (Attached). Mỗi phiên mới phải chạy lại từ đầu.

In [ ]:
!pip -q install ultralytics manga-ocr huggingface_hub
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## 1. Tải Manga109 từ Hugging Face

In [ ]:
!hf download hal-utokyo/Manga109 Manga109_released_2026_05_21.zip --repo-type dataset --local-dir /kaggle/working/m109
!cd /kaggle/working/m109 && unzip -q Manga109_released_2026_05_21.zip && rm Manga109_released_2026_05_21.zip
!ls /kaggle/working/m109 /kaggle/working/m109/*

## 2. Code + convert sang YOLO
Nếu `ls` ở trên cho thấy `images/` và `annotations/` nằm trong thư mục con khác, sửa `ROOT`.

In [ ]:
!git clone https://github.com/CPhuongg/manga-lens.git
%cd manga-lens
ROOT = "/kaggle/working/m109/Manga109_released_2026_05_21"
!PYTHONUTF8=1 python scripts/convert_manga109.py --root {ROOT} --out /kaggle/working/yolo --copy
!rm -rf /kaggle/working/m109
!cat /kaggle/working/yolo/data.yaml

## 3. Thử nhanh 1 epoch (kiểm tra pipeline), rồi train thật

In [ ]:
!python scripts/train.py --data /kaggle/working/yolo/data.yaml --model yolo11s.pt --epochs 1 --imgsz 1024 --batch 16 --name smoke

In [ ]:
!python scripts/train.py --data /kaggle/working/yolo/data.yaml --model yolo11s.pt --epochs 50 --imgsz 1024 --batch 16 --name yolo11s

## 4. Lưu kết quả (chỉ /kaggle/working được giữ)

In [ ]:
!mkdir -p /kaggle/working/out && cp -r runs /kaggle/working/out/
!ls /kaggle/working/out/runs/*/weights